# Material Transmutation - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC material transmutation results using the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Method**: Direct `material.deplete()` with multigroup flux (no geometry/transport)
- **Chain**: `chain_endf_b8.1_sfr.xml` (reduced per nuclide, level 4)
- **Energy groups**: 3-group [1e-5, 0.625, 1e5, 2e7] eV
- **Multigroup flux**: [1e12, 5e12, 1e14] n/cm²/s
- **Timesteps**: 3 × 1 hour irradiation + 2 × 15 day cooling
- **Comparison**: Nuclide atom densities at each timestep

In [1]:
# Setup and imports
import time
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"

# Transmutation data: OpenMC reads the XML chain; YAMC reads the cached arrow
# directory (auto-downloaded into ~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow/).
CHAIN_FILE_OPENMC = "/home/jon/yamc-org/yamc/crates/yamc/tests/chain_endf_b8.1_sfr.xml"
TRANSMUTATION_DATA = os.path.expanduser("~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow")

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Transmutation parameters
TRANSMUTE_ENERGY_GROUPS = [1e-5, 0.625, 1e5, 2e7]
TRANSMUTE_MULTIGROUP_FLUX = [1e12, 5e12, 1e14]
TRANSMUTE_TIMESTEPS = [3600.0, 3600.0, 3600.0, 15 * 86400.0, 15 * 86400.0]
TRANSMUTE_YAMC_SOURCE_RATES = [1.0, 1.0, 1.0, 0.0, 0.0]
TRANSMUTE_OPENMC_SOURCE_RATES = [1.06e14, 1.06e14, 1.06e14, 0.0, 0.0]
TRANSMUTE_PHASES = ["irrad", "irrad", "irrad", "cool", "cool"]
TRANSMUTE_REDUCE_LEVEL = 4

# Density threshold for comparison
DENSITY_THRESHOLD = 1e-15

In [2]:
def run_openmc_transmutation(nuclide, reduced_chain):
    """Run OpenMC material.deplete() for a single nuclide.

    Uses the full chain XML so OpenMC's fission-product-yield borrowing
    (e.g. Ra-223 → U-235) still works; the ``reduced_chain`` argument is
    ignored here for that reason.

    Returns dict with time_s and steps (list of nuclide density dicts per step).
    """
    import openmc
    import openmc.deplete

    openmc.config['cross_sections'] = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'cross_sections.xml')

    material = openmc.Material(name=nuclide)
    material.add_nuclide(nuclide, 1.0, percent_type='ao')
    material.set_density('g/cm3', 1.0)
    material.volume = 1.0
    material.temperature = 294.0
    material.depletable = True

    t0 = time.perf_counter()
    results = material.deplete(
        multigroup_flux=TRANSMUTE_MULTIGROUP_FLUX,
        energy_group_structure=TRANSMUTE_ENERGY_GROUPS,
        timesteps=TRANSMUTE_TIMESTEPS,
        source_rates=TRANSMUTE_OPENMC_SOURCE_RATES,
        timestep_units='s',
        chain_file=CHAIN_FILE_OPENMC,
    )
    elapsed = time.perf_counter() - t0

    # OpenMC returns n+1 materials (index 0 = initial state); skip index 0
    steps = []
    for mat in results[1:]:
        steps.append(mat.get_nuclide_atom_densities())

    return {'time_s': elapsed, 'steps': steps}

In [3]:
def run_yamc_transmutation(nuclide, neutron_path, reduced_chain):
    """Run YAMC material.deplete() for a single nuclide.

    Reduces the YAMC transmutation chain to the same level as OpenMC so
    only the reachable products need cross-section data.

    Returns dict with time_s and steps (list of nuclide density dicts per step).
    """
    import yamc as mc

    mc.transport_data = NUCLEAR_DATA

    # Reduce the YAMC chain to the same level as OpenMC's reduced chain.
    yamc_chain = mc.TransmutationChain(TRANSMUTATION_DATA)
    yamc_reduced = yamc_chain.reduce([nuclide], level=TRANSMUTE_REDUCE_LEVEL)

    material = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        volume=1.0,
        transmutable=True,
        name=nuclide,
    )
    material.read_nuclear_data(NUCLEAR_DATA)

    t0 = time.perf_counter()
    results = material.deplete(
        multigroup_flux=TRANSMUTE_MULTIGROUP_FLUX,
        energy_group_structure=TRANSMUTE_ENERGY_GROUPS,
        timesteps=TRANSMUTE_TIMESTEPS,
        source_rates=TRANSMUTE_YAMC_SOURCE_RATES,
        transmutation_file=yamc_reduced,
    )
    elapsed = time.perf_counter() - t0

    steps = []
    for mat in results:
        steps.append(dict(mat.nuclides))

    return {'time_s': elapsed, 'steps': steps}

In [4]:
def transmute_quality_label(max_rel_diff):
    """Assign quality label based on max relative difference (%) for transmutation."""
    if max_rel_diff < 0.1:
        return "Excellent"
    elif max_rel_diff < 1.0:
        return "Good"
    elif max_rel_diff < 5.0:
        return "Fair"
    else:
        return "Poor"


def compare_transmutation_steps(yamc_result, openmc_result):
    """Compare transmutation results step-by-step.

    Returns dict with detail_rows, summary metrics, and quality.
    """
    detail_rows = []
    shared_rel_diffs = []
    yamc_only_count = 0
    openmc_only_count = 0

    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        ymc_step = yamc_result['steps'][step_idx] if step_idx < len(yamc_result['steps']) else {}
        omc_step = openmc_result['steps'][step_idx] if step_idx < len(openmc_result['steps']) else {}

        all_products = set(ymc_step.keys()) | set(omc_step.keys())

        for product in sorted(all_products):
            y = ymc_step.get(product, 0.0)
            o = omc_step.get(product, 0.0)
            max_val = max(abs(y), abs(o))
            if max_val < DENSITY_THRESHOLD:
                continue

            y_above = abs(y) >= DENSITY_THRESHOLD
            o_above = abs(o) >= DENSITY_THRESHOLD
            shared = y_above and o_above

            if max_val > 0:
                rd = abs(y - o) / max_val * 100.0
            else:
                rd = 0.0

            detail_rows.append({
                'product': product,
                'step': step_idx + 1,
                'phase': TRANSMUTE_PHASES[step_idx],
                'yamc': y,
                'openmc': o,
                'rel_diff_pct': rd,
                'shared': shared,
            })

            if shared:
                shared_rel_diffs.append(rd)
            elif y_above and not o_above:
                yamc_only_count += 1
            else:
                openmc_only_count += 1

    if shared_rel_diffs:
        max_rel_diff_pct = max(shared_rel_diffs)
        rms_rel_diff_pct = float(np.sqrt(np.mean(np.array(shared_rel_diffs) ** 2)))
    else:
        max_rel_diff_pct = 0.0
        rms_rel_diff_pct = 0.0

    n_shared = len({dr['product'] for dr in detail_rows if dr['shared']})
    n_total = len({dr['product'] for dr in detail_rows})

    quality = transmute_quality_label(max_rel_diff_pct)
    speedup = openmc_result['time_s'] / yamc_result['time_s'] if yamc_result['time_s'] > 0 else float('inf')

    return {
        'detail_rows': detail_rows,
        'max_rel_diff_pct': max_rel_diff_pct,
        'rms_rel_diff_pct': rms_rel_diff_pct,
        'n_shared': n_shared,
        'n_total': n_total,
        'yamc_only_count': yamc_only_count,
        'openmc_only_count': openmc_only_count,
        'quality': quality,
        'speedup': speedup,
    }


def create_density_comparison_table(detail_rows, step_idx):
    """Create a density comparison table for a specific timestep."""
    step_rows = [dr for dr in detail_rows if dr['step'] == step_idx + 1]

    rows = []
    for dr in sorted(step_rows, key=lambda x: -max(abs(x['yamc']), abs(x['openmc']))):
        rows.append({
            'Nuclide': dr['product'],
            'YAMC': f"{dr['yamc']:.6e}",
            'OpenMC': f"{dr['openmc']:.6e}",
            'Rel. Diff (%)': f"{dr['rel_diff_pct']:.4f}%",
            'Status': 'Shared' if dr['shared'] else ('YAMC only' if abs(dr['yamc']) >= DENSITY_THRESHOLD else 'OpenMC only'),
        })

    return pd.DataFrame(rows)


def create_density_evolution_plot(detail_rows, nuclide):
    """Create density evolution plots for top products over timesteps."""

    # Get all products and their max densities
    product_max = {}
    for dr in detail_rows:
        p = dr['product']
        val = max(abs(dr['yamc']), abs(dr['openmc']))
        if p not in product_max or val > product_max[p]:
            product_max[p] = val

    # Top 10 products by max density (excluding the starting nuclide itself)
    top_products = sorted(
        [(p, v) for p, v in product_max.items() if p != nuclide],
        key=lambda x: -x[1]
    )[:10]

    if not top_products:
        return None

    steps = list(range(1, len(TRANSMUTE_TIMESTEPS) + 1))
    step_labels = [f"Step {s}\n({TRANSMUTE_PHASES[s-1]})" for s in steps]

    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('YAMC', 'OpenMC'),
        shared_yaxes=True,
        horizontal_spacing=0.05,
    )

    for idx, (product, _) in enumerate(top_products):
        yamc_densities = []
        openmc_densities = []
        for step in steps:
            ymc_val = 0.0
            omc_val = 0.0
            for dr in detail_rows:
                if dr['product'] == product and dr['step'] == step:
                    ymc_val = dr['yamc']
                    omc_val = dr['openmc']
                    break
            yamc_densities.append(ymc_val)
            openmc_densities.append(omc_val)

        color = colors[idx % len(colors)]

        fig.add_trace(
            go.Scatter(
                x=steps, y=yamc_densities, mode='lines+markers',
                name=product, line=dict(color=color),
                legendgroup=product,
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=steps, y=openmc_densities, mode='lines+markers',
                name=product, line=dict(color=color, dash='dash'),
                legendgroup=product, showlegend=False,
            ),
            row=1, col=2
        )

    fig.update_xaxes(title_text="Timestep", tickvals=steps, ticktext=step_labels, row=1, col=1)
    fig.update_xaxes(title_text="Timestep", tickvals=steps, ticktext=step_labels, row=1, col=2)
    fig.update_yaxes(type="log", title_text="Atom Density (atoms/b-cm)", exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", exponentformat='e', row=1, col=2)

    fig.update_layout(
        height=500,
        title=f"{nuclide} - Product Density Evolution (Top 10)",
        legend=dict(x=1.02, y=1.0),
        margin=dict(l=60, r=150, t=60, b=60),
    )

    return fig


def create_quality_metrics_table(comparison):
    """Create a summary table of quality metrics."""
    rows = [
        {'Metric': 'Max Rel. Diff (shared)', 'Value': f"{comparison['max_rel_diff_pct']:.4f}%"},
        {'Metric': 'RMS Rel. Diff (shared)', 'Value': f"{comparison['rms_rel_diff_pct']:.4f}%"},
        {'Metric': 'Shared Products', 'Value': str(comparison['n_shared'])},
        {'Metric': 'Total Products', 'Value': str(comparison['n_total'])},
        {'Metric': 'YAMC-only entries', 'Value': str(comparison['yamc_only_count'])},
        {'Metric': 'OpenMC-only entries', 'Value': str(comparison['openmc_only_count'])},
        {'Metric': 'Quality', 'Value': comparison['quality']},
        {'Metric': 'Speedup', 'Value': f"{comparison['speedup']:.1f}x"},
    ]
    return pd.DataFrame(rows)

In [5]:
# Discover nuclides: stable neutron nuclides that are in the chain.
# Restricted to stable until YAMC issue #45 is fixed (reduce() pulls in
# fission-yield products which triggers ~1000 cross-section downloads).
import openmc.deplete

chain = openmc.deplete.Chain.from_xml(CHAIN_FILE_OPENMC)
chain_stable_names = {nuc.name for nuc in chain.nuclides if nuc.half_life is None}
print(f"Chain: {len(chain.nuclides)} nuclides ({len(chain_stable_names)} stable)")

neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')
h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))

nuclides = []
for f in h5_files:
    name = os.path.splitext(os.path.basename(f))[0]
    if '_m' in name:  # skip metastable
        continue
    if name in chain_stable_names:
        nuclides.append((name, f))

if not nuclides:
    raise FileNotFoundError(
        f"No stable nuclides found in chain from {neutron_dir}\n"
        f"Please check that the chain file and nuclear data directory exist."
    )

if NUMBER_OF_SIMULATIONS:
    nuclides = nuclides[:NUMBER_OF_SIMULATIONS]

print(f"Running transmutation for {len(nuclides)} nuclides...")
for nuc, _ in nuclides:
    print(f"  {nuc}")

Chain: 3820 nuclides (259 stable)
Running transmutation for 3 nuclides...
  Ag107
  Ag109
  Al27


In [6]:
# Run all transmutations and collect results
all_results = {}

for nuclide, neutron_path in nuclides:
    print(f"\nProcessing {nuclide}...")

    # Reduce chain for this nuclide
    reduced_chain = chain.reduce([nuclide], TRANSMUTE_REDUCE_LEVEL)

    try:
        print("  Running YAMC transmute...")
        yamc_result = run_yamc_transmutation(nuclide, neutron_path, reduced_chain)
        print(f"  YAMC: {yamc_result['time_s']:.3f}s")
    except Exception as e:
        print(f"  YAMC Error: {e}")
        continue

    try:
        print("  Running OpenMC transmute...")
        openmc_result = run_openmc_transmutation(nuclide, reduced_chain)
        print(f"  OpenMC: {openmc_result['time_s']:.3f}s")
    except Exception as e:
        print(f"  OpenMC Error: {e}")
        continue

    comparison = compare_transmutation_steps(yamc_result, openmc_result)
    all_results[nuclide] = {
        'yamc': yamc_result,
        'openmc': openmc_result,
        'comparison': comparison,
    }
    print(f"  max_rel_diff={comparison['max_rel_diff_pct']:.4f}%, "
          f"quality={comparison['quality']}, "
          f"speedup={comparison['speedup']:.1f}x")

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")


Processing Ag107...
  Running YAMC transmute...
  Running OpenMC transmute...
to cache: https://github.com/fusion-neutronics/cross_section_data_endf_b8.1_arrow/releases/download/1.0.0/Ag107_m1.arrow.tar -> "/home/jon/.cache/yamc/endf-b8.1-Ag107_m1.arrow"


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Reading O16 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/O16.h5
 Reading O17 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/O17.h5
 Reading O18 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/O18.h5
 Reading F19 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/F19.h5
 Reading Ne20 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Ne20.h5


 Reading Ne21 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Ne21.h5
 Reading Ne22 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Ne22.h5
 Reading Na22 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Na22.h5
 Reading Na23 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Na23.h5


 Reading Mg24 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Mg24.h5
 Reading Mg25 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Mg25.h5
 Reading Mg26 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Mg26.h5
 Reading Al26_m1 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Al26_m1.h5


 Reading Al27 from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/neutron/Al27.h5


... (459 outputs left out)

/home/jon/yamc-org/yamc-verification/.venv/lib/python3.13/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


[openmc.deplete] t=1306800.0 s, dt=1296000.0 s, source=0.0


[openmc.deplete] t=2602800.0 (final operator evaluation)


  OpenMC: 50.476s
  max_rel_diff=0.0000%, quality=Excellent, speedup=0.6x

Completed 3 / 3 nuclides successfully.


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Density evolution plots (top 10 product nuclides)
- Quality metrics (max/RMS relative difference on shared products)
- Density comparison tables per timestep

In [7]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    comparison = results['comparison']

    # Header
    display(Markdown(f"## {nuclide}"))

    # Performance comparison
    yamc_time = results['yamc']['time_s']
    openmc_time = results['openmc']['time_s']
    display(Markdown(f"**Performance:** OpenMC: {openmc_time:.3f}s | YAMC: {yamc_time:.3f}s | Speedup: {comparison['speedup']:.1f}x"))

    # Quality metrics table
    display(Markdown("### Quality Metrics"))
    metrics_table = create_quality_metrics_table(comparison)
    display(metrics_table.style.hide(axis='index'))

    # Density evolution plot
    fig = create_density_evolution_plot(comparison['detail_rows'], nuclide)
    if fig is not None:
        fig.show()

    # Density comparison tables per timestep
    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        phase = TRANSMUTE_PHASES[step_idx]
        display(Markdown(f"### Step {step_idx + 1} ({phase})"))
        table = create_density_comparison_table(comparison['detail_rows'], step_idx)
        if not table.empty:
            display(table.style.hide(axis='index'))
        else:
            display(Markdown("*No products above density threshold.*"))

    print()  # Add spacing between nuclides

## Ag107

**Performance:** OpenMC: 51.475s | YAMC: 135.748s | Speedup: 0.4x

### Quality Metrics

Metric,Value
Max Rel. Diff (shared),0.0000%
RMS Rel. Diff (shared),0.0000%
Shared Products,16
Total Products,16
YAMC-only entries,0
OpenMC-only entries,0
Quality,Excellent
Speedup,0.4x


### Step 1 (irrad)

Nuclide,YAMC,OpenMC,Rel. Diff (%),Status
Ag107,5.633164e-03,5.633164e-03,0.0000%,Shared
Pd106,6.613573e-10,6.613573e-10,0.0000%,Shared
Ag106,5.859001e-10,5.859001e-10,0.0000%,Shared
Cd108,3.874561e-10,3.874561e-10,0.0000%,Shared
H1,2.968857e-11,2.968857e-11,0.0000%,Shared
Ag108,2.423035e-11,2.423035e-11,0.0000%,Shared
Pd107,1.703187e-11,1.703187e-11,0.0000%,Shared
Pd108,1.136644e-11,1.136644e-11,0.0000%,Shared
Ag105,8.699678e-12,8.699678e-12,0.0000%,Shared
He4,8.326704e-12,8.326705e-12,0.0000%,Shared


... (32 outputs left out)

### Step 4 (cool)

Nuclide,YAMC,OpenMC,Rel. Diff (%),Status
Al27,2.231948e-02,2.231948e-02,0.0000%,Shared
H1,4.587504e-09,4.587504e-09,0.0000%,Shared
Mg26,3.628077e-09,3.628077e-09,0.0000%,Shared
He4,1.855585e-09,1.855585e-09,0.0000%,Shared
Mg24,1.321725e-09,1.321725e-09,0.0000%,Shared
Al26,6.767141e-10,6.767141e-10,0.0000%,Shared
Na23,5.338590e-10,5.338590e-10,0.0000%,Shared
H2,2.822047e-10,2.822047e-10,0.0000%,Shared
Mg25,4.660432e-11,4.660431e-11,0.0000%,Shared
H3,4.579931e-11,4.579931e-11,0.0000%,Shared


### Step 5 (cool)

Nuclide,YAMC,OpenMC,Rel. Diff (%),Status
Al27,2.231948e-02,2.231948e-02,0.0000%,Shared
H1,4.587504e-09,4.587504e-09,0.0000%,Shared
Mg26,3.628077e-09,3.628077e-09,0.0000%,Shared
He4,1.855585e-09,1.855585e-09,0.0000%,Shared
Mg24,1.321726e-09,1.321726e-09,0.0000%,Shared
Al26,6.767141e-10,6.767141e-10,0.0000%,Shared
Na23,5.338590e-10,5.338590e-10,0.0000%,Shared
H2,2.822047e-10,2.822047e-10,0.0000%,Shared
Mg25,4.660432e-11,4.660431e-11,0.0000%,Shared
H3,4.569361e-11,4.569361e-11,0.0000%,Shared
